# Generate Latent Embeddings for Cell Images

This notebook generates latent embeddings for cell images using the OpenPhenom MAE model.
It saves embeddings along with drug and cell pair information for each dataset.

In [1]:
import os
import json
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
from PIL import Image
import pandas as pd
from tqdm import tqdm

from huggingface_mae import MAEModel

/home/shpark/.conda/envs/cellmaes/lib/python3.10/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
# Configuration
DATA_DIR = Path("/home/shpark/prj-molrepr/data/cell-image")
OUTPUT_DIR = Path("/home/shpark/prj-molrepr/data/cell-image/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# MODEL_NAME = "recursionpharma/OpenPhenom"
MODEL_NAME = "./"
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# Load the model
print("Loading model...")
model = MAEModel.from_pretrained(MODEL_NAME)
model = model.eval()
model = model.to(DEVICE)
print("Model loaded successfully!")

Loading model...


/home/shpark/prj-molrepr/mae_microscopy/huggingface_mae.py:303: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(modelpath, map_location="cpu")


Model loaded successfully!


## Helper Functions

In [4]:
def load_image(image_path: Path) -> torch.Tensor:
    """Load an image and convert to tensor format expected by the model.
    
    Expected format: (C, H, W) where C is number of channels (1-11),
    H and W are height and width (should be 256x256 for best results).
    Supports: .npy, .tif, .tiff, .jp2, .png, .jpg, .jpeg
    """
    # Handle numpy arrays
    if image_path.suffix == '.npy':
        img = np.load(image_path)
        if len(img.shape) == 2:
            img = img[np.newaxis, :, :]
        elif len(img.shape) == 3 and img.shape[0] > 11:
            # If shape is (H, W, C), transpose
            img = np.transpose(img, (2, 0, 1))
    # Handle TIFF files
    elif image_path.suffix in ['.tif', '.tiff']:
        from skimage import io
        img = io.imread(str(image_path))
        if len(img.shape) == 3 and img.shape[2] <= 11:
            img = np.transpose(img, (2, 0, 1))
        elif len(img.shape) == 2:
            img = img[np.newaxis, :, :]
    # Handle JP2 format
    elif image_path.suffix == '.jp2':
        im = Image.open(image_path)
        img = np.array(im)
        if len(img.shape) == 2:
            img = img[np.newaxis, :, :]
        elif len(img.shape) == 3 and img.shape[2] <= 11:
            img = np.transpose(img, (2, 0, 1))
    # Handle regular image formats
    else:
        img = np.array(Image.open(image_path))
        if len(img.shape) == 2:
            img = img[np.newaxis, :, :]
        elif len(img.shape) == 3 and img.shape[2] <= 11:
            img = np.transpose(img, (2, 0, 1))
    
    # Ensure image is uint8 and in correct range
    if img.dtype != np.uint8:
        img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        img = img.astype(np.uint8)
    
    # Resize to 256x256 if needed
    if img.shape[1] != 256 or img.shape[2] != 256:
        from skimage.transform import resize
        resized_channels = []
        for c in range(img.shape[0]):
            resized = resize(img[c], (256, 256), preserve_range=True, anti_aliasing=True)
            resized_channels.append(resized)
        img = np.stack(resized_channels, axis=0)
        img = img.astype(np.uint8)
    
    return torch.tensor(img, dtype=torch.uint8)

In [5]:
def generate_embeddings_batch(
    model: MAEModel,
    images: List[torch.Tensor],
    device: str = "cpu"
) -> np.ndarray:
    """Generate embeddings for a batch of images."""
    # Stack images into a batch
    batch_tensor = torch.stack(images).to(device)
    
    with torch.no_grad():
        embeddings = model.predict(batch_tensor)
    
    return embeddings.cpu().numpy()

In [6]:
def process_dataset(
    dataset_name: str,
    image_dir: Path,
    metadata_file: Optional[Path] = None,
    model: Optional[MAEModel] = None,
    device: str = "cpu",
    batch_size: int = 8
) -> Dict:
    """Process a dataset and generate embeddings.
    
    Args:
        dataset_name: Name of the dataset
        image_dir: Directory containing images
        metadata_file: Optional CSV/JSON file with drug and cell pair information
        model: The MAE model
        device: Device to run inference on
        batch_size: Batch size for processing
    
    Returns:
        Dictionary containing embeddings and metadata
    """
    print(f"\nProcessing dataset: {dataset_name}")
    
    # Find all image files
    image_extensions = ['.tif', '.tiff', '.jp2', '.png', '.jpg', '.jpeg']
    image_files = []
    for ext in image_extensions:
        image_files.extend(list(image_dir.glob(f"*{ext}")))
        image_files.extend(list(image_dir.glob(f"**/*{ext}")))
    
    if len(image_files) == 0:
        print(f"No images found in {image_dir}")
        return None
    
    print(f"Found {len(image_files)} images")
    
    # Load metadata if available
    metadata = None
    if metadata_file and metadata_file.exists():
        if metadata_file.suffix == '.csv':
            metadata = pd.read_csv(metadata_file)
        elif metadata_file.suffix == '.json':
            with open(metadata_file, 'r') as f:
                metadata = json.load(f)
    
    # Process images in batches
    all_embeddings = []
    all_metadata = []
    
    for i in tqdm(range(0, len(image_files), batch_size), desc="Processing batches"):
        batch_files = image_files[i:i+batch_size]
        batch_images = []
        
        for img_file in batch_files:
            try:
                img_tensor = load_image(img_file)
                batch_images.append(img_tensor)
                
                # Extract metadata for this image
                img_meta = {
                    'image_path': str(img_file),
                    'image_name': img_file.name,
                    'dataset': dataset_name
                }
                
                # Try to match with metadata file
                if metadata is not None and isinstance(metadata, pd.DataFrame):
                    # Try to match by filename or other identifier
                    # Adjust this based on your metadata structure
                    if 'image_path' in metadata.columns:
                        match = metadata[metadata['image_path'] == str(img_file)]
                    elif 'image_name' in metadata.columns:
                        match = metadata[metadata['image_name'] == img_file.name]
                    else:
                        match = pd.DataFrame()
                    
                    if not match.empty:
                        row = match.iloc[0]
                        # Extract drug and cell information
                        for col in ['drug', 'cell', 'cell_line', 'compound', 'smiles', 'well', 'plate']:
                            if col in row:
                                img_meta[col] = str(row[col])
                
                all_metadata.append(img_meta)
                
            except Exception as e:
                print(f"Error loading {img_file}: {e}")
                continue
        
        if len(batch_images) > 0:
            # Generate embeddings
            embeddings = generate_embeddings_batch(model, batch_images, device)
            all_embeddings.append(embeddings)
    
    if len(all_embeddings) == 0:
        print(f"No embeddings generated for {dataset_name}")
        return None
    
    # Concatenate all embeddings
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    
    print(f"Generated embeddings shape: {all_embeddings.shape}")
    
    return {
        'embeddings': all_embeddings,
        'metadata': all_metadata,
        'dataset_name': dataset_name
    }

## Generate Embeddings for All Datasets

In [7]:
# Find all dataset directories
datasets = []
if DATA_DIR.exists():
    for item in DATA_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            datasets.append(item.name)

print(f"Found {len(datasets)} datasets: {datasets}")

# If no datasets found, create a toy example
if len(datasets) == 0:
    print("\nNo datasets found. Creating toy example...")
    toy_dir = DATA_DIR / "toy_dataset"
    toy_dir.mkdir(parents=True, exist_ok=True)
    
    # Create some dummy images
    print("Generating toy images...")
    for i in range(10):
        # Create a random 6-channel image (CellPainting format)
        img = np.random.randint(0, 255, size=(6, 256, 256), dtype=np.uint8)
        
        # Save as numpy file (you can change this to your preferred format)
        np.save(toy_dir / f"cell_image_{i}.npy", img)
    
    # Create toy metadata
    toy_metadata = pd.DataFrame({
        'image_name': [f"cell_image_{i}.npy" for i in range(10)],
        'drug': [f"drug_{i % 3}" for i in range(10)],
        'cell': [f"cell_{i % 2}" for i in range(10)],
        'well': [f"A{i+1:02d}" for i in range(10)],
        'plate': [f"plate_{i // 5 + 1}" for i in range(10)]
    })
    toy_metadata.to_csv(toy_dir / "metadata.csv", index=False)
    
    datasets = ["toy_dataset"]
    print(f"Created toy dataset with 10 images")

Found 3 datasets: ['cpjump1', 'embeddings', 'rxrxr1']


In [ ]:
# Note: load_image function is already defined above and handles all formats including .npy

In [8]:
# Process each dataset
results = {}

for dataset_name in datasets:
    dataset_dir = DATA_DIR / dataset_name
    metadata_file = dataset_dir / "metadata.csv"
    
    if not metadata_file.exists():
        metadata_file = dataset_dir / "metadata.json"
    
    result = process_dataset(
        dataset_name=dataset_name,
        image_dir=dataset_dir,
        metadata_file=metadata_file if metadata_file.exists() else None,
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE
    )
    
    if result is not None:
        results[dataset_name] = result


Processing dataset: cpjump1
No images found in /home/shpark/prj-molrepr/data/cell-image/cpjump1

Processing dataset: embeddings
No images found in /home/shpark/prj-molrepr/data/cell-image/embeddings

Processing dataset: rxrxr1
No images found in /home/shpark/prj-molrepr/data/cell-image/rxrxr1


## Save Embeddings and Metadata

In [ ]:
# Save embeddings for each dataset
for dataset_name, result in results.items():
    print(f"\nSaving embeddings for {dataset_name}...")
    
    dataset_output_dir = OUTPUT_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save embeddings as numpy array
    embeddings_path = dataset_output_dir / "embeddings.npy"
    np.save(embeddings_path, result['embeddings'])
    print(f"Saved embeddings to {embeddings_path}")
    print(f"Embeddings shape: {result['embeddings'].shape}")
    
    # Save metadata as JSON
    metadata_path = dataset_output_dir / "metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(result['metadata'], f, indent=2)
    print(f"Saved metadata to {metadata_path}")
    
    # Also save as CSV for easier inspection
    metadata_df = pd.DataFrame(result['metadata'])
    metadata_csv_path = dataset_output_dir / "metadata.csv"
    metadata_df.to_csv(metadata_csv_path, index=False)
    print(f"Saved metadata CSV to {metadata_csv_path}")
    
    # Save summary
    summary = {
        'dataset_name': dataset_name,
        'num_images': len(result['metadata']),
        'embedding_dim': result['embeddings'].shape[1],
        'embeddings_path': str(embeddings_path),
        'metadata_path': str(metadata_path)
    }
    
    summary_path = dataset_output_dir / "summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Saved summary to {summary_path}")

## Verify Saved Data

In [ ]:
# Load and verify saved embeddings
for dataset_name in results.keys():
    print(f"\n=== {dataset_name} ===")
    
    dataset_output_dir = OUTPUT_DIR / dataset_name
    
    # Load embeddings
    embeddings = np.load(dataset_output_dir / "embeddings.npy")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Embeddings dtype: {embeddings.dtype}")
    print(f"Embeddings stats: min={embeddings.min():.4f}, max={embeddings.max():.4f}, mean={embeddings.mean():.4f}")
    
    # Load metadata
    metadata_df = pd.read_csv(dataset_output_dir / "metadata.csv")
    print(f"\nMetadata columns: {list(metadata_df.columns)}")
    print(f"Number of samples: {len(metadata_df)}")
    
    # Show first few rows
    print("\nFirst few rows:")
    print(metadata_df.head())
    
    # Show drug and cell pair statistics
    if 'drug' in metadata_df.columns and 'cell' in metadata_df.columns:
        print("\nDrug-Cell pair counts:")
        print(metadata_df.groupby(['drug', 'cell']).size())